# Nepali Cultural Dress — Image Captioning
### ResNet-50 (from scratch) + Bahdanau Attention + GRU

Trains an encoder–decoder captioning model **entirely from scratch** (no pretrained
weights) on the Nepali dress dataset.

**How to use on Kaggle**
1. Create a Kaggle Dataset containing `captions.csv` and the `nepali_dresses_augmented/`
   folder (keep the folder structure so the relative `image_path` values in the CSV resolve).
2. Add that dataset to the notebook (right panel → *Add Input*).
3. In **Section 1 (Config)** set `CFG.DATA_DIR` to your dataset's mount path
   (e.g. `/kaggle/input/nepali-dress-captioning`).
4. Turn on GPU (Settings → Accelerator → GPU) and *Run All*.

> **Everything you'd want to tune lives in the single Config cell below.**
> You should not need to edit any other cell for normal experiments.


## 1. Configuration  ⚙️ *(edit only here)*

In [ ]:
import os

class CFG:
    # ---------------- Reproducibility ----------------
    SEED = 42

    # ---------------- Paths (Kaggle) ----------------
    # DATA_DIR must contain captions.csv AND the nepali_dresses_augmented/ folder.
    DATA_DIR   = "."                                          # <-- CHANGE THIS
    CSV_DIR    = "."                                           # dir containing captions.csv
    CSV_NAME   = "captions.csv"
    CAPTION_COL = "caption_en"        # "caption_en" or "caption_ne"
    OUTPUT_DIR = "model1_output"                               # writable dir for checkpoints/logs

    # ---------------- Data & split ----------------
    VAL_FRACTION    = 0.10
    TEST_FRACTION   = 0.10
    SPLIT_BY_SOURCE = True   # keep an image + its augmentations on the SAME side (no leakage)
    IMAGE_SIZE      = 224
    NORM_MEAN = [0.485, 0.456, 0.406]
    NORM_STD  = [0.229, 0.224, 0.225]
    USE_TRAIN_AUG = True     # light on-the-fly aug (flip + jitter) during training

    # ---------------- Vocabulary ----------------
    MIN_WORD_FREQ   = 2      # words rarer than this become <unk>
    MAX_CAPTION_LEN = 22     # tokens incl <start>/<end>; longer captions are truncated

    # ---------------- Model ----------------
    ENCODER_PRETRAINED = False   # False => ResNet-50 trained FROM SCRATCH (your requirement)
    ENC_IMAGE_SIZE = 14          # encoder output grid (14x14 spatial locations)
    ENCODER_DIM    = 2048        # ResNet-50 final channels (do not change)
    EMBED_DIM      = 256         # word embedding size
    ATTENTION_DIM  = 256         # Bahdanau attention hidden size
    DECODER_DIM    = 512         # GRU hidden size
    DROPOUT        = 0.5

    # ---------------- Training ----------------
    EPOCHS      = 100
    BATCH_SIZE  = 32
    ENCODER_LR  = 1e-4
    DECODER_LR  = 4e-4
    GRAD_CLIP   = 5.0
    ALPHA_C     = 1.0        # doubly-stochastic attention regularization (0 to disable)
    NUM_WORKERS = 0

    # ---------------- Early stopping & checkpoints ----------------
    EARLY_STOPPING = True
    PATIENCE       = 8         # epochs without val improvement before stopping
    MONITOR        = "bleu4"   # "bleu4" (higher better) or "loss" (lower better)
    CKPT_BEST      = "best_model.pth"
    CKPT_LAST      = "last_model.pth"

    # ---------------- Resume (Kaggle sessions are time-limited) ----------------
    RESUME      = False
    RESUME_PATH = "/kaggle/working/last_model.pth"

    # ---------------- Inference / evaluation ----------------
    BEAM_SIZE = 3

# quick sanity print
print({k: v for k, v in vars(CFG).items() if not k.startswith("__")})

## 2. Imports & setup

In [ ]:
import os, re, json, random, time
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print("Torch:", torch.__version__, "| Device:", DEVICE)

## 3. Load captions & leakage-safe split

We split by **source image** so an original and all of its augmentations
(`12.jpg`, `12_aug1.jpg`, …) always land in the same split. This prevents the
model from "seeing" a validation/test image during training.

In [ ]:
csv_path = os.path.join(CFG.CSV_DIR, CFG.CSV_NAME)
df = pd.read_csv(csv_path)
assert CFG.CAPTION_COL in df.columns, f"{CFG.CAPTION_COL} not in {list(df.columns)}"
print("total rows:", len(df))

def parse_source(path):
    cls = os.path.basename(os.path.dirname(path))
    fn  = os.path.basename(path)
    m = re.match(r"(\d+)(?:_aug\d+)?\.jpg", fn, re.I)   # strip _augN suffix
    stem = m.group(1) if m else os.path.splitext(fn)[0]
    return cls, f"{cls}/{stem}"

df["cls"], df["source"] = zip(*df["image_path"].map(parse_source))

# choose grouping key: by source (no leakage) or per-row
groups = df["source"] if CFG.SPLIT_BY_SOURCE else pd.Series(df.index.astype(str), index=df.index)
uniq = sorted(groups.unique())
rng = random.Random(CFG.SEED); rng.shuffle(uniq)
n = len(uniq)
n_test = int(n * CFG.TEST_FRACTION)
n_val  = int(n * CFG.VAL_FRACTION)
test_g = set(uniq[:n_test])
val_g  = set(uniq[n_test:n_test + n_val])

def which(g):
    if g in test_g: return "test"
    if g in val_g:  return "val"
    return "train"

df["split"] = groups.map(which)
print(df["split"].value_counts())
print("\nper-class / split:\n", df.groupby(["split", "cls"]).size().unstack(fill_value=0))

## 4. Vocabulary (built from the **train** split only)

In [ ]:
PAD, START, END, UNK = "<pad>", "<start>", "<end>", "<unk>"

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9ऀ-ॿ]+", " ", text)   # keep latin + devanagari
    return text.split()

class Vocab:
    def __init__(self, freqs, min_freq):
        self.itos = [PAD, START, END, UNK]
        for w, c in sorted(freqs.items(), key=lambda x: (-x[1], x[0])):
            if c >= min_freq and w not in (PAD, START, END, UNK):
                self.itos.append(w)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, text, max_len):
        toks = [START] + tokenize(text)[: max_len - 2] + [END]
        length = len(toks)
        ids = [self.stoi.get(t, self.stoi[UNK]) for t in toks]
        ids += [self.stoi[PAD]] * (max_len - length)
        return ids, length

freqs = Counter()
for c in df.loc[df.split == "train", CFG.CAPTION_COL]:
    freqs.update(tokenize(c))
vocab = Vocab(freqs, CFG.MIN_WORD_FREQ)
pad_idx, start_idx, end_idx = vocab.stoi[PAD], vocab.stoi[START], vocab.stoi[END]
print("vocab size:", len(vocab))
print("sample words:", vocab.itos[4:24])

## 5. Transforms, Dataset & DataLoaders

In [ ]:
_aug = [T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.1)] if CFG.USE_TRAIN_AUG else []
train_tf = T.Compose([T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)), *_aug,
                      T.ToTensor(), T.Normalize(CFG.NORM_MEAN, CFG.NORM_STD)])
eval_tf  = T.Compose([T.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
                      T.ToTensor(), T.Normalize(CFG.NORM_MEAN, CFG.NORM_STD)])

class CaptionDataset(Dataset):
    def __init__(self, frame, vocab, transform):
        self.frame = frame.reset_index(drop=True)
        self.vocab = vocab
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(os.path.join(CFG.DATA_DIR, row["image_path"])).convert("RGB")
        img = self.transform(img)
        ids, length = self.vocab.encode(row[CFG.CAPTION_COL], CFG.MAX_CAPTION_LEN)
        return img, torch.tensor(ids), torch.tensor(length)

def make_loader(split, transform, shuffle):
    ds = CaptionDataset(df[df.split == split], vocab, transform)
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=shuffle,
                      num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=shuffle)

train_loader = make_loader("train", train_tf, True)
val_loader   = make_loader("val",   eval_tf, False)
test_loader  = make_loader("test",  eval_tf, False)
print("batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))

## 6. Model

### 6a. Encoder — ResNet-50 (from scratch)
`weights=None` means the backbone is randomly initialized and learned from your
data. We drop the classifier head and keep the `14×14×2048` spatial feature grid
that the attention module attends over.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, encoded_image_size=14, pretrained=False):
        super().__init__()
        weights = torchvision.models.ResNet50_Weights.DEFAULT if pretrained else None
        resnet = torchvision.models.resnet50(weights=weights)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])   # drop avgpool+fc
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))
    def forward(self, images):
        x = self.backbone(images)              # [B, 2048, H/32, W/32]
        x = self.adaptive_pool(x)              # [B, 2048, enc, enc]
        x = x.permute(0, 2, 3, 1)              # [B, enc, enc, 2048]
        return x

### 6b. Bahdanau (additive) Attention

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att    = nn.Linear(attention_dim, 1)
        self.tanh    = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)
    def forward(self, encoder_out, decoder_hidden):
        # encoder_out: [B, num_pixels, encoder_dim]  decoder_hidden: [B, decoder_dim]
        att1 = self.encoder_att(encoder_out)                    # [B, num_pixels, att_dim]
        att2 = self.decoder_att(decoder_hidden).unsqueeze(1)    # [B, 1, att_dim]
        att  = self.full_att(self.tanh(att1 + att2)).squeeze(2) # [B, num_pixels]
        alpha = self.softmax(att)                               # [B, num_pixels]
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1) # [B, encoder_dim]
        return context, alpha

### 6c. Decoder — GRU with attention

In [ ]:
class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, decoder_dim, attention_dim,
                 encoder_dim=2048, dropout=0.5):
        super().__init__()
        self.vocab_size  = vocab_size
        self.encoder_dim = encoder_dim
        self.attention = BahdanauAttention(encoder_dim, decoder_dim, attention_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout   = nn.Dropout(dropout)
        self.gru       = nn.GRUCell(embed_dim + encoder_dim, decoder_dim)
        self.init_h    = nn.Linear(encoder_dim, decoder_dim)   # init GRU state from image
        self.f_beta    = nn.Linear(decoder_dim, encoder_dim)   # attention gating scalar
        self.sigmoid   = nn.Sigmoid()
        self.fc        = nn.Linear(decoder_dim, vocab_size)
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0); self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, encoder_out):
        return torch.tanh(self.init_h(encoder_out.mean(dim=1)))

    def forward(self, encoder_out, encoded_captions, caption_lengths):
        B = encoder_out.size(0)
        encoder_out = encoder_out.view(B, -1, self.encoder_dim)   # [B, num_pixels, enc_dim]
        num_pixels = encoder_out.size(1)

        # sort by decreasing caption length so we can prune finished sequences
        caption_lengths, sort_ind = caption_lengths.sort(dim=0, descending=True)
        encoder_out      = encoder_out[sort_ind]
        encoded_captions = encoded_captions[sort_ind]
        embeddings = self.embedding(encoded_captions)            # [B, max_len, embed_dim]

        h = self.init_hidden_state(encoder_out)                  # [B, decoder_dim]
        decode_lengths = (caption_lengths - 1).tolist()          # predict up to <end>
        max_dec = max(decode_lengths)
        preds  = torch.zeros(B, max_dec, self.vocab_size, device=encoder_out.device)
        alphas = torch.zeros(B, max_dec, num_pixels,      device=encoder_out.device)

        for t in range(max_dec):
            bs = sum(l > t for l in decode_lengths)              # active (still-decoding) samples
            context, alpha = self.attention(encoder_out[:bs], h[:bs])
            gate = self.sigmoid(self.f_beta(h[:bs]))             # [bs, encoder_dim]
            context = gate * context
            gru_in = torch.cat([embeddings[:bs, t, :], context], dim=1)
            h = self.gru(gru_in, h[:bs])                         # teacher forcing
            preds[:bs, t, :]  = self.fc(self.dropout(h))
            alphas[:bs, t, :] = alpha
        return preds, encoded_captions, decode_lengths, alphas, sort_ind

## 7. Training utilities

In [ ]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val = val; self.sum += val * n; self.count += n; self.avg = self.sum / self.count

def clip_gradient(optimizer, grad_clip):
    for group in optimizer.param_groups:
        for p in group["params"]:
            if p.grad is not None:
                p.grad.data.clamp_(-grad_clip, grad_clip)

def save_checkpoint(path, epoch, best_score):
    torch.save({
        "epoch": epoch,
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "enc_opt": enc_opt.state_dict(),
        "dec_opt": dec_opt.state_dict(),
        "best_score": best_score,
        "vocab_itos": vocab.itos,
        "config": {k: v for k, v in vars(CFG).items() if not k.startswith("__")},
    }, path)

criterion = nn.CrossEntropyLoss().to(DEVICE)

## 8. Train / validate functions

In [ ]:
def run_epoch(loader, train):
    encoder.train(train); decoder.train(train)
    losses = AverageMeter()
    references, hypotheses = [], []
    from tqdm.auto import tqdm
    loader_bar = tqdm(loader, desc=f"{'train' if train else 'val':5s}", leave=False)
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, caps, lengths in loader_bar:
            imgs, caps, lengths = imgs.to(DEVICE), caps.to(DEVICE), lengths.to(DEVICE)
            feats = encoder(imgs)
            preds, caps_sorted, decode_lengths, alphas, _ = decoder(feats, caps, lengths)
            targets = caps_sorted[:, 1:]

            preds_p   = nn.utils.rnn.pack_padded_sequence(preds,   decode_lengths, batch_first=True).data
            targets_p = nn.utils.rnn.pack_padded_sequence(targets, decode_lengths, batch_first=True).data
            loss = criterion(preds_p, targets_p)
            if CFG.ALPHA_C > 0:
                loss = loss + CFG.ALPHA_C * ((1.0 - alphas.sum(dim=1)) ** 2).mean()

            if train:
                dec_opt.zero_grad(); enc_opt.zero_grad()
                loss.backward()
                if CFG.GRAD_CLIP:
                    clip_gradient(dec_opt, CFG.GRAD_CLIP); clip_gradient(enc_opt, CFG.GRAD_CLIP)
                dec_opt.step(); enc_opt.step()
            losses.update(loss.item(), sum(decode_lengths))
            loader_bar.set_postfix(loss=f"{losses.avg:.3f}")

            if not train:
                words = preds.argmax(dim=2)
                for j in range(caps_sorted.size(0)):
                    ref = [w for w in caps_sorted[j].tolist() if w not in (pad_idx, start_idx, end_idx)]
                    hyp = [w for w in words[j][:decode_lengths[j]].tolist() if w not in (pad_idx, start_idx, end_idx)]
                    references.append([ref]); hypotheses.append(hyp)

    if train:
        return losses.avg, None
    smooth = SmoothingFunction().method1
    bleu4 = corpus_bleu(references, hypotheses, smoothing_function=smooth)
    return losses.avg, bleu4

## 9. Build models, optimizers (+ optional resume)

In [ ]:
encoder = Encoder(CFG.ENC_IMAGE_SIZE, CFG.ENCODER_PRETRAINED).to(DEVICE)
decoder = DecoderWithAttention(len(vocab), CFG.EMBED_DIM, CFG.DECODER_DIM,
                               CFG.ATTENTION_DIM, CFG.ENCODER_DIM, CFG.DROPOUT).to(DEVICE)

enc_opt = torch.optim.Adam(encoder.parameters(), lr=CFG.ENCODER_LR)
dec_opt = torch.optim.Adam(decoder.parameters(), lr=CFG.DECODER_LR)

n_params = sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in decoder.parameters())
print(f"trainable params: {n_params/1e6:.1f}M")

start_epoch = 0
best_score  = -1e9 if CFG.MONITOR == "bleu4" else 1e9
epochs_no_improve = 0
if CFG.RESUME and os.path.exists(CFG.RESUME_PATH):
    ck = torch.load(CFG.RESUME_PATH, map_location=DEVICE)
    encoder.load_state_dict(ck["encoder"]); decoder.load_state_dict(ck["decoder"])
    enc_opt.load_state_dict(ck["enc_opt"]); dec_opt.load_state_dict(ck["dec_opt"])
    start_epoch = ck["epoch"] + 1; best_score = ck["best_score"]
    print(f"Resumed from epoch {start_epoch} (best={best_score:.4f})")

## 10. Training loop — early stopping + checkpoints

- `last_model.pth` is written every epoch (use it with `CFG.RESUME` if the Kaggle
  session times out).
- `best_model.pth` is written whenever the monitored metric improves.
- Training stops after `CFG.PATIENCE` epochs with no improvement.

In [ ]:
def is_better(new, best):
    return new > best if CFG.MONITOR == "bleu4" else new < best

from tqdm.auto import tqdm
history = []
epoch_bar = tqdm(range(start_epoch, CFG.EPOCHS), desc="Epochs")
for epoch in epoch_bar:
    t0 = time.time()
    train_loss, _        = run_epoch(train_loader, train=True)
    val_loss, val_bleu4  = run_epoch(val_loader,   train=False)
    score = val_bleu4 if CFG.MONITOR == "bleu4" else val_loss
    dt = time.time() - t0
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_loss": val_loss, "val_bleu4": val_bleu4})
    epoch_bar.set_postfix(train=train_loss, val=val_loss, bleu=val_bleu4)
    print(f"Epoch {epoch+1:02d}/{CFG.EPOCHS} | train {train_loss:.3f} | "
          f"val {val_loss:.3f} | BLEU-4 {val_bleu4:.4f} | {dt:.0f}s")

    save_checkpoint(os.path.join(CFG.OUTPUT_DIR, CFG.CKPT_LAST), epoch, best_score)
    if is_better(score, best_score):
        best_score = score; epochs_no_improve = 0
        save_checkpoint(os.path.join(CFG.OUTPUT_DIR, CFG.CKPT_BEST), epoch, best_score)
        print(f"   new best ({CFG.MONITOR} = {best_score:.4f}) -> saved best_model.pth")
    else:
        epochs_no_improve += 1
        print(f"   no improvement ({epochs_no_improve}/{CFG.PATIENCE})")
        if CFG.EARLY_STOPPING and epochs_no_improve >= CFG.PATIENCE:
            print("Early stopping triggered."); break

with open(os.path.join(CFG.OUTPUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)
print("done. best score:", best_score)

## 11. Training curves

In [ ]:
import matplotlib.pyplot as plt
if history:
    ep = [h["epoch"] + 1 for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, [h["train_loss"] for h in history], label="train")
    ax[0].plot(ep, [h["val_loss"] for h in history], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, [h["val_bleu4"] for h in history], color="green")
    ax[1].set_title("Val BLEU-4"); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.show()

## 12. Inference — beam-search caption generation

Loads `best_model.pth` and generates a caption for any image with beam search.

In [ ]:
def load_best():
    p = os.path.join(CFG.OUTPUT_DIR, CFG.CKPT_BEST)
    if os.path.exists(p):
        ck = torch.load(p, map_location=DEVICE)
        encoder.load_state_dict(ck["encoder"]); decoder.load_state_dict(ck["decoder"])
        print("loaded", p)
load_best()

@torch.no_grad()
def caption_image(image_path, beam_size=CFG.BEAM_SIZE, max_len=CFG.MAX_CAPTION_LEN):
    encoder.eval(); decoder.eval()
    img = eval_tf(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    enc = encoder(img).view(1, -1, CFG.ENCODER_DIM)          # [1, num_pixels, enc_dim]
    k = beam_size
    num_pixels = enc.size(1)
    enc = enc.expand(k, num_pixels, CFG.ENCODER_DIM)
    seqs = torch.full((k, 1), start_idx, dtype=torch.long, device=DEVICE)
    top_scores = torch.zeros(k, 1, device=DEVICE)
    h = decoder.init_hidden_state(enc)
    complete_seqs, complete_scores = [], []
    V = len(vocab)

    for step in range(1, max_len + 1):
        emb = decoder.embedding(seqs[:, -1])                 # [k, embed]
        context, _ = decoder.attention(enc, h)
        context = decoder.sigmoid(decoder.f_beta(h)) * context
        h = decoder.gru(torch.cat([emb, context], dim=1), h)
        scores = F.log_softmax(decoder.fc(h), dim=1)         # [k, V]
        scores = top_scores.expand_as(scores) + scores
        if step == 1:
            top_scores, top_words = scores[0].topk(k, 0)
        else:
            top_scores, top_words = scores.view(-1).topk(k, 0)
        prev = torch.div(top_words, V, rounding_mode="floor")
        next_word = top_words % V
        seqs = torch.cat([seqs[prev], next_word.unsqueeze(1)], dim=1)

        incomplete = [i for i, w in enumerate(next_word) if w.item() != end_idx]
        complete   = [i for i in range(len(next_word)) if i not in incomplete]
        for i in complete:
            complete_seqs.append(seqs[i].tolist()); complete_scores.append(top_scores[i].item())
        k -= len(complete)
        if k == 0:
            break
        seqs = seqs[incomplete]; h = h[prev[incomplete]]; enc = enc[prev[incomplete]]
        top_scores = top_scores[incomplete].unsqueeze(1)

    if not complete_seqs:                                    # fallback: nothing hit <end>
        complete_seqs = seqs.tolist(); complete_scores = top_scores.squeeze(1).tolist()
    best = complete_seqs[int(np.argmax(complete_scores))]
    words = [vocab.itos[w] for w in best if w not in (pad_idx, start_idx, end_idx)]
    return " ".join(words)

## 13. Show predictions on random test images

In [ ]:
sample = df[df.split == "test"].sample(min(4, (df.split == "test").sum()), random_state=CFG.SEED)
plt.figure(figsize=(12, 10))
for i, (_, row) in enumerate(sample.iterrows()):
    p = os.path.join(CFG.DATA_DIR, row["image_path"])
    pred = caption_image(p, CFG.BEAM_SIZE)
    ax = plt.subplot(2, 2, i + 1)
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
    ax.set_title(f"pred: {pred}\ngt: {row[CFG.CAPTION_COL][:70]}", fontsize=9)
plt.tight_layout(); plt.show()

## 14. Final BLEU on the test set (beam search)

This is the "real" free-running metric (no teacher forcing). It is slower because
it generates each caption; use `limit` to sample a subset for a quick estimate.

In [ ]:
@torch.no_grad()
def evaluate_bleu(split="test", beam_size=CFG.BEAM_SIZE, limit=None):
    rows = df[df.split == split]
    if limit:
        rows = rows.sample(min(limit, len(rows)), random_state=CFG.SEED)
    refs, hyps = [], []
    for _, row in rows.iterrows():
        gt   = tokenize(row[CFG.CAPTION_COL])
        pred = caption_image(os.path.join(CFG.DATA_DIR, row["image_path"]), beam_size).split()
        refs.append([gt]); hyps.append(pred)
    smooth = SmoothingFunction().method1
    b1 = corpus_bleu(refs, hyps, weights=(1, 0, 0, 0),          smoothing_function=smooth)
    b2 = corpus_bleu(refs, hyps, weights=(0.5, 0.5, 0, 0),      smoothing_function=smooth)
    b3 = corpus_bleu(refs, hyps, weights=(1/3, 1/3, 1/3, 0),    smoothing_function=smooth)
    b4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    print(f"{split}: BLEU-1 {b1:.4f} | BLEU-2 {b2:.4f} | BLEU-3 {b3:.4f} | BLEU-4 {b4:.4f}")
    return b1, b2, b3, b4

# quick estimate on 200 test images; set limit=None for the full test set
evaluate_bleu("test", CFG.BEAM_SIZE, limit=200)

## Notes & expectations

- **From scratch is hard on this data.** ResNet-50 has ~25M params and is being
  trained on ~3.6k images; expect the encoder to overfit and BLEU-4 to be modest.
  That is inherent to the "no pretrained weights" constraint, not a bug.
- If you are ever allowed to relax it, set `CFG.ENCODER_PRETRAINED = True` — it is
  the single biggest quality lever for small datasets.
- The val BLEU used for early stopping is *teacher-forced* (cheap). Section 14 gives
  the true beam-search BLEU. If you prefer, set `CFG.MONITOR = "loss"`.
- Checkpoints + `history.json` are written to `/kaggle/working` and appear under the
  notebook's **Output** tab after the session ends.


In [ ]:
# ===== Report Visualizations =====
# Run this AFTER training completes and inference cell has been executed.
# Requires: encoder, decoder, vocab, df, history, caption_image, evaluate_bleu

import matplotlib.pyplot as plt
import numpy as np
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from PIL import Image

has_history = len(history) > 0 if history else False

fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3)

# ----------------------------------------------------------
# 1. Training curves (loss + BLEU-4)
# ----------------------------------------------------------
ax1 = fig.add_subplot(gs[0, :2])
if has_history:
    ep = [h["epoch"] + 1 for h in history]
    ax1.plot(ep, [h["train_loss"] for h in history], "o-", label="Train Loss", color="#e74c3c")
    ax1.plot(ep, [h["val_loss"] for h in history], "s-", label="Val Loss", color="#3498db")
    ax1.set_ylabel("Loss", color="#c0392b")
    ax1.tick_params(axis="y", labelcolor="#c0392b")
    ax1.legend(loc="upper left")
    ax1.set_title("Training & Validation Loss", fontsize=13, fontweight="bold")

    ax1b = ax1.twinx()
    ax1b.plot(ep, [h["val_bleu4"] for h in history], "d-", label="Val BLEU-4", color="#2ecc71", linewidth=2)
    ax1b.set_ylabel("BLEU-4", color="#27ae60")
    ax1b.tick_params(axis="y", labelcolor="#27ae60")
    ax1b.legend(loc="upper right")
    ax1.set_xlabel("Epoch")
    ax1.grid(True, alpha=0.3)

# ----------------------------------------------------------
# 2. Final BLEU-1..4 bar chart
# ----------------------------------------------------------
ax2 = fig.add_subplot(gs[0, 2])
try:
    b_scores = evaluate_bleu("test", CFG.BEAM_SIZE, limit=200)
    b_labels = ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]
    colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"]
    bars = ax2.barh(b_labels, b_scores, color=colors, edgecolor="white", height=0.6)
    for bar, score in zip(bars, b_scores):
        ax2.text(score + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{score:.4f}", va="center", fontsize=10)
    ax2.set_xlim(0, max(b_scores) + 0.05)
    ax2.set_title("Test Set BLEU Scores", fontsize=13, fontweight="bold")
    ax2.grid(True, axis="x", alpha=0.3)
except Exception as e:
    ax2.text(0.5, 0.5, f"BLEU eval failed:\n{e}", ha="center", va="center", transform=ax2.transAxes, fontsize=10)

# ----------------------------------------------------------
# 3. Per-class BLEU-4
# ----------------------------------------------------------
ax3 = fig.add_subplot(gs[1, :])
try:
    classes = sorted(df[df.split == "test"]["cls"].unique())
    cls_scores = []
    for cls in classes:
        rows = df[(df.split == "test") & (df["cls"] == cls)]
        refs, hyps = [], []
        for _, row in rows.iterrows():
            gt = tokenize(row[CFG.CAPTION_COL])
            pred = caption_image(os.path.join(CFG.DATA_DIR, row["image_path"]), CFG.BEAM_SIZE).split()
            refs.append([gt]); hyps.append(pred)
        smooth = SmoothingFunction().method1
        b4 = corpus_bleu(refs, hyps, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
        cls_scores.append(b4)

    cls_labels = [c.replace("_", " ").title() for c in classes]
    bar_colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))
    bars = ax3.bar(cls_labels, cls_scores, color=bar_colors, edgecolor="gray", linewidth=0.5)
    for bar, score in zip(bars, cls_scores):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f"{score:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax3.set_title("Per-Class BLEU-4 on Test Set", fontsize=13, fontweight="bold")
    ax3.set_ylabel("BLEU-4")
    ax3.set_ylim(0, max(cls_scores) + 0.05)
    ax3.grid(True, axis="y", alpha=0.3)
    plt.setp(ax3.get_xticklabels(), rotation=30, ha="right", fontsize=9)
except Exception as e:
    ax3.text(0.5, 0.5, f"Per-class BLEU failed:\n{e}", ha="center", va="center", transform=ax3.transAxes, fontsize=10)

# ----------------------------------------------------------
# 4. Attention map visualization
# ----------------------------------------------------------
ax4 = fig.add_subplot(gs[2, 0])
ax5 = fig.add_subplot(gs[2, 1])
ax6 = fig.add_subplot(gs[2, 2])
try:
    test_samples = df[df.split == "test"].sample(3, random_state=42)
    for ax_i, (_, row) in zip([ax4, ax5, ax6], test_samples.iterrows()):
        img_path = os.path.join(CFG.DATA_DIR, row["image_path"])
        img = Image.open(img_path).convert("RGB")
        pred = caption_image(img_path, CFG.BEAM_SIZE)
        gt = row[CFG.CAPTION_COL]
        ax_i.imshow(img)
        ax_i.axis("off")
        ax_i.set_title(f"GT: {gt[:50]}\nPred: {pred[:50]}", fontsize=8, pad=10)
except Exception as e:
    for ax_i in [ax4, ax5, ax6]:
        ax_i.text(0.5, 0.5, f"Samples failed:\n{e}", ha="center", va="center", transform=ax_i.transAxes, fontsize=10)
        ax_i.axis("off")

plt.suptitle("Nepali Cultural Dress Captioning — Report Visualizations",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# 5. Summary metrics table
# ----------------------------------------------------------
print("=" * 60)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 60)
if has_history:
    best_idx = np.argmax([h["val_bleu4"] for h in history])
    best_epoch = history[best_idx]["epoch"] + 1
    print(f"Best epoch:           {best_epoch}")
    print(f"Best val BLEU-4:      {history[best_idx]['val_bleu4']:.4f}")
    print(f"Final train loss:     {history[-1]['train_loss']:.4f}")
    print(f"Final val loss:       {history[-1]['val_loss']:.4f}")
    print(f"Total epochs trained: {len(history)}")
try:
    b1, b2, b3, b4 = b_scores
    print(f"Test BLEU-1:           {b1:.4f}")
    print(f"Test BLEU-2:           {b2:.4f}")
    print(f"Test BLEU-3:           {b3:.4f}")
    print(f"Test BLEU-4:           {b4:.4f}")
except:
    pass
print(f"Vocabulary size:      {len(vocab)}")
print(f"Dataset size:         {len(df)} ({len(df[df.split=='train'])} train / {len(df[df.split=='val'])} val / {len(df[df.split=='test'])} test)")
print(f"Classes:              {len(df['cls'].unique())}")
print("=" * 60)
